In [1]:
from pathlib import Path
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import trange

In [2]:
DATA_DIR = Path("/content/drive/MyDrive/SLM/Data")

In [3]:
class GPTConfig:
    def __init__(self, vocab_size, block_size,
                 n_layer=8, n_head=8, n_embd=512, dropout=0.1):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout

In [4]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.key = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.query = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.value = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.attn_drop = nn.Dropout(config.dropout)
        self.resid_drop = nn.Dropout(config.dropout)

        # causal mask
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
                .view(1, 1, config.block_size, config.block_size)
        )

    def forward(self, x):
        B, T, C = x.size()
        k = self.key(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = self.query(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = self.value(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.proj(y))
        return y

In [5]:
class Block(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [6]:
class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size

        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, ix = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

In [7]:
import torch
from tokenizers import Tokenizer

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [9]:
train_ids = torch.load(DATA_DIR / "train_ids.pt")
val_ids = torch.load(DATA_DIR / "val_ids.pt")
tokenizer = Tokenizer.from_file(str(DATA_DIR / "tokenizer.json"))

In [10]:
block_size = 128
batch_size = 16
lr = 3e-4
max_iters = 5_000
eval_interval = 500

In [11]:
def get_batch(split="train"):
    data = train_ids if split == "train" else val_ids
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

In [12]:
config = GPTConfig(
    vocab_size=tokenizer.get_vocab_size(),
    block_size=block_size,
    n_layer=4,
    n_head=4,
    n_embd=512,
    dropout=0.1,
)

In [13]:
model = GPT(config).to(device)

In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [ ]:
for it in trange(max_iters):
    model.train()
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if it % eval_interval == 0:
        model.eval()
        with torch.no_grad():
            xb, yb = get_batch("val")
            _, val_loss = model(xb, yb)

        print(f"iter {it} | train loss {loss.item():.4f} | val loss {val_loss.item():.4f}")


torch.save(model.state_dict(), "slm_tinystories_personachat.pt")
print("Model saved.")

  0%|          | 1/5000 [00:02<3:21:08,  2.41s/it]

iter 0 | train loss 9.1084 | val loss 8.3459


 10%|█         | 501/5000 [12:16<1:54:05,  1.52s/it]

iter 500 | train loss 4.0925 | val loss 4.2398


 20%|██        | 1001/5000 [24:37<1:56:58,  1.76s/it]

...

 70%|███████   | 3501/5000 [1:26:07<38:38,  1.55s/it]

iter 3500 | train loss 3.3188 | val loss 3.4154


 80%|████████  | 4001/5000 [1:38:24<30:57,  1.86s/it]

iter 4000 | train loss 3.5805 | val loss 3.2297


 89%|████████▉ | 4440/5000 [1:49:08<14:32,  1.56s/it]